<a href="https://colab.research.google.com/github/joserlandero/DatosMasivos/blob/main/Tarea_4_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea 4/5 - Uso de MLIB en Spark

**Nombre del Alumno:** Jose Landero

**Dataset:** Costo de siniestro de accidentes vehiculares.
- Se hace un cambio en el dataset ya que los datos no estaban bien estructurados para algún modelo de regresión.

**Objetivo:** Crear un modelo de Regresión Lineal para predecir el coste de los siniestros

In [ ]:
!pip install spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.7/351.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
import spark
import pandas as pd

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tarea") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/content/siniestros.csv")
)



In [ ]:
df.printSchema()
print(f"total de registros: {df.count()}")

# Nuestra variable objetivo es claimAmount

root
 |-- _c0: integer (nullable = true)
 |-- IDpol: double (nullable = true)
 |-- ClaimNb: double (nullable = true)
 |-- Exposure: double (nullable = true)
 |-- Area: string (nullable = true)
 |-- VehPower: double (nullable = true)
 |-- VehAge: double (nullable = true)
 |-- DrivAge: double (nullable = true)
 |-- BonusMalus: double (nullable = true)
 |-- VehBrand: string (nullable = true)
 |-- VehGas: string (nullable = true)
 |-- Density: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- ClaimAmount: double (nullable = true)

total de registros: 26444


In [ ]:
df.limit(5).toPandas()

,_c0,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount
0,0,139.0,1.0,0.75,F,7.0,1.0,61.0,50.0,B12,Regular,27000.0,R11,303.00
1,1,190.0,1.0,0.14,B,12.0,5.0,50.0,60.0,B12,Diesel,56.0,R25,1981.84
2,2,414.0,1.0,0.14,E,4.0,0.0,36.0,85.0,B12,Regular,4792.0,R11,1456.55
3,3,424.0,2.0,0.62,F,10.0,0.0,51.0,100.0,B12,Regular,27000.0,R11,989.64
4,4,424.0,2.0,0.62,F,10.0,0.0,51.0,100.0,B12,Regular,27000.0,R11,9844.36


In [ ]:
df.count()

26444

## Análisis de datos

In [ ]:
# Análisis de cada variable

df.toPandas().describe(include='all')

,_c0,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount
count,26444.000000,2.644400e+04,26444.000000,26444.000000,26444,26444.000000,26444.000000,26444.000000,26444.000000,26444,26444,26444.000000,26444,2.644400e+04
unique,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN,11,2,NaN,22,NaN
top,NaN,NaN,NaN,NaN,C,NaN,NaN,NaN,NaN,B1,Diesel,NaN,R24,NaN
freq,NaN,NaN,NaN,NaN,7093,NaN,NaN,NaN,NaN,6898,13450,NaN,6475,NaN
mean,13221.500000,2.280004e+06,1.139427,0.691791,NaN,6.464415,7.355090,45.121502,65.231054,NaN,NaN,2015.302942,NaN,2.265513e+03
std,7633.869595,1.583004e+06,0.617563,0.313213,NaN,2.017260,5.165475,14.694677,20.143480,NaN,NaN,4162.639153,NaN,2.937103e+04
min,0.000000,1.390000e+02,1.000000,0.002740,NaN,4.000000,0.000000,18.000000,50.000000,NaN,NaN,2.000000,NaN,1.000000e+00
25%,6610.750000,1.086381e+06,1.000000,0.450000,NaN,5.000000,3.000000,34.000000,50.000000,NaN,NaN,115.000000,NaN,6.859925e+02
50%,13221.500000,2.133756e+06,1.000000,0.760000,NaN,6.000000,7.000000,45.000000,55.000000,NaN,NaN,524.500000,NaN,1.172000e+03
75%,19832.250000,3.183953e+06,1.000000,1.000000,NaN,7.000000,11.000000,54.000000,76.000000,NaN,NaN,2252.000000,NaN,1.212385e+03


In [ ]:
df.toPandas().info()

# No se observan valores nullos.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26444 entries, 0 to 26443
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   _c0          26444 non-null  int32  
 1   IDpol        26444 non-null  float64
 2   ClaimNb      26444 non-null  float64
 3   Exposure     26444 non-null  float64
 4   Area         26444 non-null  object 
 5   VehPower     26444 non-null  float64
 6   VehAge       26444 non-null  float64
 7   DrivAge      26444 non-null  float64
 8   BonusMalus   26444 non-null  float64
 9   VehBrand     26444 non-null  object 
 10  VehGas       26444 non-null  object 
 11  Density      26444 non-null  float64
 12  Region       26444 non-null  object 
 13  ClaimAmount  26444 non-null  float64
dtypes: float64(9), int32(1), object(4)
memory usage: 2.7+ MB


### Variables categoricas

Estas variables las convertiremos a One Hot Encoder para poder realizar la regresión lineal.

In [ ]:
df.select('Area').distinct().toPandas()

,Area
0,F
1,E
2,B
3,D
4,C
5,A


In [ ]:
df.select('VehBrand').distinct().toPandas()

,VehBrand
0,B4
1,B6
2,B2
3,B13
4,B1
5,B3
6,B10
7,B5
8,B14
9,B12


In [ ]:
df.select('Region').distinct().toPandas()

,Region
0,R82
1,R23
2,R73
3,R21
4,R43
5,R72
6,R94
7,R24
8,R53
9,R54


### Revisamos cuanto varia el costo promedio de un siniestro deendiendo de la variable categórica para ver que tanto impacta esa variable en el costo

In [ ]:
import pyspark.sql.functions as F

# Ejemplo: ¿Importa la columna 'Condicion_Clima'?
df.groupBy("Region") \
  .agg(
      F.count("*").alias("Cantidad_Accidentes"),
      F.mean("ClaimAmount").alias("Costo_Promedio")
  ).orderBy(F.desc("Cantidad_Accidentes")).show()

+------+-------------------+------------------+
|Region|Cantidad_Accidentes|    Costo_Promedio|
+------+-------------------+------------------+
|   R24|               6475|2945.9154223937962|
|   R82|               4233|2428.1369832270316|
|   R93|               2986|2301.0799732083115|
|   R11|               2591|1761.1817290621455|
|   R53|               1871|2085.0086851950887|
|   R52|               1576|1631.3229251269033|
|   R91|               1068|1901.6021348314584|
|   R72|               1055| 1816.245061611373|
|   R31|                944| 1731.662838983049|
|   R54|                800| 1631.384724999999|
|   R41|                468| 2210.045512820513|
|   R25|                452|2057.1843141592917|
|   R73|                369|1461.7486991869926|
|   R26|                345| 1844.342144927536|
|   R22|                314|2189.8881210191075|
|   R23|                220|1488.6071818181824|
|   R74|                197|1422.7672081218277|
|   R83|                141|1304.7860283

In [ ]:
import pyspark.sql.functions as F

# Ejemplo: ¿Importa la columna 'Condicion_Clima'?
df.groupBy("Area") \
  .agg(
      F.count("*").alias("Cantidad_Accidentes"),
      F.mean("ClaimAmount").alias("Costo_Promedio")
  ).orderBy(F.desc("Cantidad_Accidentes")).show()

+----+-------------------+------------------+
|Area|Cantidad_Accidentes|    Costo_Promedio|
+----+-------------------+------------------+
|   C|               7093|2060.0694247850165|
|   D|               6458| 2243.186924744522|
|   E|               6122|2126.3355602744323|
|   A|               3364|2300.7225535077405|
|   B|               2633| 3370.292259779725|
|   F|                774|1524.0393023255797|
+----+-------------------+------------------+



### Conversion de variables categoricas a Dummies

Esto se hara solamente para **Area** y **VehBrand**

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

# 1. Definimos las columnas de entrada y los nombres de las columnas de salida
input_cols = ["Area", "VehBrand",'VehGas']

# 2. StringIndexer: Convierte el texto ("Area A", "BMW", etc.) en números (0, 1, 2...)
# Creamos una lista de nombres para los índices (ej: Area_Index, VeBrand_Index)
index_cols = [c + "_Index" for c in input_cols]
indexer = StringIndexer(inputCols=input_cols, outputCols=index_cols, handleInvalid="keep")

# 3. OneHotEncoder: Convierte esos números en vectores binarios (Dummies)
# Creamos una lista de nombres para los vectores finales (ej: Area_Vec, VeBrand_Vec)
vec_cols = [c + "_Vec" for c in input_cols]
encoder = OneHotEncoder(inputCols=index_cols, outputCols=vec_cols, dropLast=True)

# 4. Usamos un Pipeline para ejecutar ambos pasos en orden
pipeline = Pipeline(stages=[indexer, encoder])

# Entrenamos el modelo de transformación y lo aplicamos al DataFrame
model = pipeline.fit(df)
df_final = model.transform(df)

#Dropeamos ciertas columnas
df_final = df_final.drop('_c0','IDpol','Area','Area_Index','VehBrand','VehBrand_Index','ClaimNb','Region','VehGas','VehGas_Index')
# Ver el resultado (mostramos la original y la convertida)
df_final.select("Area_Vec", "VehBrand_Vec").show(10)

+-------------+--------------+
|     Area_Vec|  VehBrand_Vec|
+-------------+--------------+
|(6,[5],[1.0])|(11,[2],[1.0])|
|(6,[4],[1.0])|(11,[2],[1.0])|
|(6,[2],[1.0])|(11,[2],[1.0])|
|(6,[5],[1.0])|(11,[2],[1.0])|
|(6,[5],[1.0])|(11,[2],[1.0])|
|(6,[3],[1.0])|(11,[2],[1.0])|
|(6,[1],[1.0])|(11,[2],[1.0])|
|(6,[1],[1.0])|(11,[2],[1.0])|
|(6,[2],[1.0])|(11,[2],[1.0])|
|(6,[2],[1.0])|(11,[0],[1.0])|
+-------------+--------------+
only showing top 10 rows


In [ ]:
df_final.limit(5).toPandas()

,Exposure,VehPower,VehAge,DrivAge,BonusMalus,Density,ClaimAmount,Area_Vec,VehBrand_Vec,VehGas_Vec
0,0.75,7.0,1.0,61.0,50.0,27000.0,303.00,"(0.0, 0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0)"
1,0.14,12.0,5.0,50.0,60.0,56.0,1981.84,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(1.0, 0.0)"
2,0.14,4.0,0.0,36.0,85.0,4792.0,1456.55,"(0.0, 0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0)"
3,0.62,10.0,0.0,51.0,100.0,27000.0,989.64,"(0.0, 0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0)"
4,0.62,10.0,0.0,51.0,100.0,27000.0,9844.36,"(0.0, 0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0)"


## Modelo de regresion lineal

Variable Y = **ClaimAmount**

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
df_final = df_final.drop('ClaimNb','Density','Exposure')
print(df_final.columns)
# 1. Definimos la variable objetivo y las features
target = "ClaimAmount"
feature_cols = [col for col in df_final.columns if col != target]

# 2. Ensamblamos todas las features en un solo vector
#    VectorAssembler maneja tanto columnas escalares como vectores (Area_Vec, etc.)
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
df_assembled = assembler.transform(df_final)

# 3. Split 80/20
train, test = df_assembled.randomSplit([0.8, 0.2], seed=42)

print(f"Registros entrenamiento: {train.count()}")
print(f"Registros prueba: {test.count()}")

# 4. Creamos y entrenamos el modelo de regresión lineal
lr = LinearRegression(
    featuresCol="features",
    labelCol=target)
modelo = lr.fit(train)

# 5. Predicciones sobre el conjunto de prueba
predicciones = modelo.transform(test)
predicciones.select(target, "prediction").show(10)

# 6. Métricas de evaluación
metricas = {}
for metrica in ["r2", "rmse", "mae"]:
    evaluator = RegressionEvaluator(
        labelCol=target,
        predictionCol="prediction",
        metricName=metrica
    )
    metricas[metrica] = evaluator.evaluate(predicciones)

print(f"\n--- Métricas en Test ---")
print(f"R²:   {metricas['r2']:.4f}")
print(f"RMSE: {metricas['rmse']:.4f}")
print(f"MAE:  {metricas['mae']:.4f}")

# 7. Resumen del modelo (coeficientes e intercepto)
print(f"\nIntercepto: {modelo.intercept:.4f}")
print(f"Número de coeficientes: {len(modelo.coefficients)}")


['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'ClaimAmount', 'Area_Vec', 'VehBrand_Vec', 'VehGas_Vec']
Registros entrenamiento: 21166
Registros prueba: 5278
+-----------+------------------+
|ClaimAmount|        prediction|
+-----------+------------------+
|    1697.37|2587.3705808807254|
|     2408.0| 2055.963829933958|
|    4163.88|2760.3412636539747|
|     1204.0| 3211.342243327659|
|     1204.0|2784.5098211720883|
|     1204.0|2729.9839210927325|
|    2177.32|2805.7818407353648|
|      83.51| 2770.576044045972|
|    1003.15| 2056.648681865481|
|     370.02| 2419.063154314946|
+-----------+------------------+
only showing top 10 rows

--- Métricas en Test ---
R²:   0.0001
RMSE: 56732.5583
MAE:  2677.7252

Intercepto: 1963.9939
Número de coeficientes: 23


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Variables
target = "ClaimAmount"
feature_cols = [col for col in df_final.columns if col != target]

# 2. Ensamblamos features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
df_assembled = assembler.transform(df_final)

# 3. Split 80/20
train, test = df_assembled.randomSplit([0.8, 0.2], seed=42)

# 4. Random Forest
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target,
    numTrees=100,
    maxDepth=10,
    seed=42
)
modelo_rf = rf.fit(train)

# 5. Predicciones
predicciones = modelo_rf.transform(test)
predicciones.select(target, "prediction").show(10)

# 6. Métricas
metricas = {}
for metrica in ["r2", "rmse", "mae"]:
    evaluator = RegressionEvaluator(
        labelCol=target,
        predictionCol="prediction",
        metricName=metrica
    )
    metricas[metrica] = evaluator.evaluate(predicciones)

print(f"\n--- Métricas en Test ---")
print(f"R²:   {metricas['r2']:.4f}")
print(f"RMSE: {metricas['rmse']:.4f}")
print(f"MAE:  {metricas['mae']:.4f}")



+-----------+------------------+
|ClaimAmount|        prediction|
+-----------+------------------+
|     1172.0|3741.8567241570317|
|    1633.71|10745.339994915312|
|   11936.45| 5584.658944559426|
|    2397.27| 2841.439052939002|
|    1247.16|2774.4779419846086|
|     107.88| 4415.057604594978|
|    5009.89|2592.3859616818486|
|      81.29| 3699.035813078314|
|     1204.0| 2133.546347169692|
|    5234.12|2716.7562750651473|
+-----------+------------------+
only showing top 10 rows

--- Métricas en Test ---
R²:   -0.0039
RMSE: 57213.8060
MAE:  2856.5122


Observamos que el modelo lineal y el arbol de desición no desempeña bien con el conjunto de datos